In [1]:
import numpy as np
import torch

SEED = 1234
torch.manual_seed(SEED)
torch.backends.cudnn.deterministic = True

# 0. Choosing "teacher" model

In [2]:
checkpoint = "bert-base-uncased"
task_name  = "mrpc"

# 1. Loading our mrpc part of the GLUE dataset

In [3]:
from datasets import load_dataset
from transformers import AutoTokenizer, DataCollatorWithPadding

raw_datasets = load_dataset("glue", task_name)
tokenizer = AutoTokenizer.from_pretrained(checkpoint)

/usr/local/lib/python3.10/dist-packages/tqdm/auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm


In [4]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [5]:
raw_datasets['train']

Dataset({
    features: ['sentence1', 'sentence2', 'label', 'idx'],
    num_rows: 3668
})

In [6]:
from datasets import DatasetDict

In [7]:
raw_datasets = DatasetDict({
    "train": raw_datasets['train'],
    "validation": raw_datasets['validation'],
    "test": raw_datasets['test']
})

In [8]:
raw_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx'],
        num_rows: 1725
    })
})

In [9]:
raw_train_dataset = raw_datasets["train"]
raw_train_dataset[0]

{'sentence1': 'Amrozi accused his brother , whom he called " the witness " , of deliberately distorting his evidence .',
 'sentence2': 'Referring to him as only " the witness " , Amrozi accused his brother of deliberately distorting his evidence .',
 'label': 1,
 'idx': 0}

In [10]:
raw_train_dataset[5]['sentence1'], raw_train_dataset[5]['sentence2']

('Revenue in the first quarter of the year dropped 15 percent from the same period a year earlier .',
 "With the scandal hanging over Stewart 's company , revenue the first quarter of the year dropped 15 percent from the same period a year earlier .")

In [11]:
raw_train_dataset[5]['label']

1

In [12]:
raw_train_dataset[5]['idx']

5

In [13]:
raw_train_dataset.features

{'sentence1': Value('string'),
 'sentence2': Value('string'),
 'label': ClassLabel(names=['not_equivalent', 'equivalent']),
 'idx': Value('int32')}

# 2. Preprocess

In [14]:
def tokenize_function(example):
    return tokenizer(example["sentence1"], example["sentence2"], truncation=True)

tokenized_datasets = raw_datasets.map(tokenize_function, batched=True)
tokenized_datasets

DatasetDict({
    train: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 3668
    })
    validation: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 408
    })
    test: Dataset({
        features: ['sentence1', 'sentence2', 'label', 'idx', 'input_ids', 'token_type_ids', 'attention_mask'],
        num_rows: 1725
    })
})

# 3. Preparing for Training

In [15]:
from transformers import DataCollatorWithPadding

data_collator = DataCollatorWithPadding(tokenizer=tokenizer)

In [16]:
tokenized_datasets = tokenized_datasets.remove_columns(["sentence1", "sentence2", "idx"])
tokenized_datasets = tokenized_datasets.rename_column("label", "labels")
tokenized_datasets.set_format("torch")
tokenized_datasets["train"].column_names

['labels', 'input_ids', 'token_type_ids', 'attention_mask']

In [17]:
from torch.utils.data import DataLoader

train_dataloader = DataLoader(
    tokenized_datasets["train"], shuffle=True, batch_size=32, collate_fn=data_collator
)

val_dataloader = DataLoader(
    tokenized_datasets["validation"], batch_size=32, collate_fn=data_collator
)

eval_dataloader = DataLoader(
    tokenized_datasets["test"], batch_size=32, collate_fn=data_collator
)

In [18]:
for batch in train_dataloader:
    break
{k: v.shape for k, v in batch.items()}

{'labels': torch.Size([32]),
 'input_ids': torch.Size([32, 100]),
 'token_type_ids': torch.Size([32, 100]),
 'attention_mask': torch.Size([32, 100])}

# 4. Load Model

In [19]:
import sys
sys.path.insert(0, '../')

In [20]:
from Bert_model.modeling_bert import BertForSequenceClassification

In [21]:
# id2label, label2id dicts for the outputs for the model
labels = tokenized_datasets["train"].features["labels"].names
num_labels = len(labels)
label2id, id2label = dict(), dict()
for i, label in enumerate(labels):
    label2id[label] = str(i)
    id2label[str(i)] = label

In [22]:
model = BertForSequenceClassification.from_pretrained(
    checkpoint,
    num_labels=num_labels,
    id2label=id2label,
    label2id=label2id,
    output_hidden_states=True,
    output_attentions=True
)

Some weights of BertForSequenceClassification were not initialized from the model checkpoint at bert-base-uncased and are newly initialized: ['classifier.bias', 'classifier.weight']
You should probably TRAIN this model on a down-stream task to be able to use it for predictions and inference.


In [23]:
model.set_use_module_grafting(False)
model.set_use_scc_status(False)

In [24]:
outputs = model(**batch)
print(outputs.loss, outputs.logits.shape)

tensor(1.2996, grad_fn=<NllLossBackward0>) torch.Size([32, 2])


In [25]:
device = torch.device("cuda:1") if torch.cuda.is_available() else torch.device("cpu")
model.to(device)

device

device(type='cuda', index=1)

### Load Trained Weights

In [26]:
load_path = '../glue_fine_tune/weights/'
best_weight = torch.load(load_path + f'bert-{task_name}.pt', map_location=device)
model.load_state_dict(best_weight['model_state_dict'])

<All keys matched successfully>

In [27]:
from train_eval_func import eval_loop

In [28]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

### Keep Copy of Model Layers 

In [29]:
import copy

In [30]:
original_layers = copy.deepcopy(model.bert.encoder.layer)

In [31]:
from train_eval_func import get_primary_metric
target_metric = get_primary_metric(task_name)

# 5. Drop Layer Experiment 

### Top Layer Drop

In [32]:
top_drop_track = []

In [33]:
for iter in range(6):
    model.bert.encoder.layer = model.bert.encoder.layer[:-1]
    print("Number of Layers Remaining:", len(model.bert.encoder.layer))
    print("")

    eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
    top_drop_track.append(eval_metric)
    
    print("  Metrics:")
    for metric_name, value in eval_metric.items():
        marker = "★" if metric_name == target_metric else " "
        print(f"    {marker} {metric_name}: {value:.4f}")

Number of Layers Remaining: 11

  Metrics:
      accuracy: 0.8799
    ★ f1: 0.9171
Number of Layers Remaining: 10

  Metrics:
      accuracy: 0.8725
    ★ f1: 0.9100
Number of Layers Remaining: 9

  Metrics:
      accuracy: 0.7892
    ★ f1: 0.8259
Number of Layers Remaining: 8

  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000
Number of Layers Remaining: 7

  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000
Number of Layers Remaining: 6

  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000


In [34]:
import torch.nn as nn

In [35]:
model.bert.encoder.layer = nn.ModuleList(
    [copy.deepcopy(layer) for layer in original_layers]
)

model.config.num_hidden_layers = len(model.bert.encoder.layer)

In [36]:
torch.cuda.empty_cache()

In [37]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

### Similarity Layer Drop

In [38]:
from CKA import CKAEvaluator

cka_eval = CKAEvaluator(device)

In [39]:
cka_max_iter =150 

In [40]:
similarity_drop_track = []

In [42]:
for iter in range(6):
    cls_reps_similarity = cka_eval.pairwise(
        model, 
        train_dataloader, 
        device, 
        only_cls_token=True, 
        max_iter=cka_max_iter
    )
    stats = cka_eval.similarity_stats(cls_reps_similarity)
    
    print("  Similarity Stats:")
    print(f"    Average: {stats['average']:.6f}")
    print(f"    Highest: {stats['max']:.6f}")
    print(f"    Lowest:  {stats['min']:.6f}")

    merge_layer = stats['argmax']
    print("Merge Layer:", merge_layer)    

    model.bert.encoder.layer = nn.ModuleList(
        [layer for i, layer in enumerate(model.bert.encoder.layer) if i != merge_layer]
    )
    model.config.num_hidden_layers = len(model.bert.encoder.layer)

    print("Number of Layers Remaining:", len(model.bert.encoder.layer))
    print("")

    eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
    similarity_drop_track.append(eval_metric)
    
    print("  Metrics:")
    for metric_name, value in eval_metric.items():
        marker = "★" if metric_name == target_metric else " "
        print(f"    {marker} {metric_name}: {value:.4f}")
    
    print("")
    print("")

CKA Evaluation: 100%|██████████| 115/115 [00:10<00:00, 10.92it/s]


  Similarity Stats:
    Average: 0.899780
    Highest: 0.993581
    Lowest:  0.726629
Merge Layer: 9
Number of Layers Remaining: 11

  Metrics:
      accuracy: 0.8799
    ★ f1: 0.9142




CKA Evaluation: 100%|██████████| 115/115 [00:09<00:00, 12.56it/s]


  Similarity Stats:
    Average: 0.890170
    Highest: 0.987652
    Lowest:  0.724432
Merge Layer: 8
Number of Layers Remaining: 10

  Metrics:
      accuracy: 0.7574
    ★ f1: 0.7925




CKA Evaluation: 100%|██████████| 115/115 [00:07<00:00, 14.95it/s]


...

  Similarity Stats:
    Average: 0.876100
    Highest: 0.984421
    Lowest:  0.722253
Merge Layer: 5
Number of Layers Remaining: 8

  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000




CKA Evaluation: 100%|██████████| 115/115 [00:05<00:00, 21.03it/s]


  Similarity Stats:
    Average: 0.851922
    Highest: 0.959973
    Lowest:  0.785674
Merge Layer: 1
Number of Layers Remaining: 7

  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000




CKA Evaluation: 100%|██████████| 115/115 [00:04<00:00, 24.83it/s]


  Similarity Stats:
    Average: 0.834847
    Highest: 0.916755
    Lowest:  0.712388
Merge Layer: 3
Number of Layers Remaining: 6

  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000




In [43]:
model.bert.encoder.layer = nn.ModuleList(
    [copy.deepcopy(layer) for layer in original_layers]
)

model.config.num_hidden_layers = len(model.bert.encoder.layer)

In [44]:
torch.cuda.empty_cache()

In [45]:
eval_loop(model, val_dataloader, task_name, device)[0]

{'accuracy': 0.8676470588235294, 'f1': 0.9093959731543624}

### Merge

In [47]:
from structure_prune_helper import compute_importance_scores, register_importance_masks, remove_importance_masks

In [48]:
register_importance_masks(model, device, register_heads=True, register_ffn=True)

  Layer 0: Registered masks (12 heads, 3072 neurons)
  Layer 1: Registered masks (12 heads, 3072 neurons)
  Layer 2: Registered masks (12 heads, 3072 neurons)
  Layer 3: Registered masks (12 heads, 3072 neurons)
  Layer 4: Registered masks (12 heads, 3072 neurons)
  Layer 5: Registered masks (12 heads, 3072 neurons)
  Layer 6: Registered masks (12 heads, 3072 neurons)
  Layer 7: Registered masks (12 heads, 3072 neurons)
  Layer 8: Registered masks (12 heads, 3072 neurons)
  Layer 9: Registered masks (12 heads, 3072 neurons)
  Layer 10: Registered masks (12 heads, 3072 neurons)
  Layer 11: Registered masks (12 heads, 3072 neurons)


BertForSequenceClassification(
  (bert): BertModel(
    (embeddings): BertEmbeddings(
      (word_embeddings): Embedding(30522, 768, padding_idx=0)
      (position_embeddings): Embedding(512, 768)
      (token_type_embeddings): Embedding(2, 768)
      (LayerNorm): LayerNorm((768,), eps=1e-12, elementwise_affine=True)
      (dropout): Dropout(p=0.1, inplace=False)
    )
    (encoder): BertEncoder(
      (layer): ModuleList(
        (0-11): 12 x BertLayer(
          (attention): BertAttention(
            (self): BertSelfAttention(
              (query): Linear(in_features=768, out_features=768, bias=True)
              (key): Linear(in_features=768, out_features=768, bias=True)
              (value): Linear(in_features=768, out_features=768, bias=True)
              (dropout): Dropout(p=0.1, inplace=False)
            )
            (output): BertSelfOutput(
              (dense): Linear(in_features=768, out_features=768, bias=True)
              (LayerNorm): LayerNorm((768,), eps=1e-12,

In [49]:
from layer_merge_helper import merge_mha, merge_ff

In [50]:
merge_track = []

In [51]:
for iter in range(6):
    cls_reps_similarity = cka_eval.pairwise(
        model, 
        train_dataloader, 
        device, 
        only_cls_token=True, 
        max_iter=cka_max_iter
    )
    stats = cka_eval.similarity_stats(cls_reps_similarity)
    
    print("  Similarity Stats:")
    print(f"    Average: {stats['average']:.6f}")
    print(f"    Highest: {stats['max']:.6f}")
    print(f"    Lowest:  {stats['min']:.6f}")

    merge_layer = stats['argmax']
    print("Merge Layer:", merge_layer)    

    head_importance, neuron_importance = compute_importance_scores(model, train_dataloader, device, compute_heads=True, compute_ffn=True)

    head_imp1, indices1 = torch.sort(head_importance[merge_layer], descending=True)
    head_imp2, indices2 = torch.sort(head_importance[merge_layer+1], descending=True)

    neuron_imp1, indices1 = torch.sort(neuron_importance[merge_layer], descending=True)        
    neuron_imp2, indices1 = torch.sort(neuron_importance[merge_layer+1], descending=True)
    
    merge_mha(model, model.bert.encoder.layer[merge_layer], model.bert.encoder.layer[merge_layer+1], head_imp1, head_imp2, device, 4)
    merge_ff(model, model.bert.encoder.layer[merge_layer], model.bert.encoder.layer[merge_layer+1], neuron_imp1, neuron_imp2, device, extra_neurons=700)

    del model.bert.encoder.layer[merge_layer]

    eval_metric = eval_loop(model, val_dataloader, task_name, device)[0]
    print("  Metrics:")
    for metric_name, value in eval_metric.items():
        marker = "★" if metric_name == target_metric else " "
        print(f"    {marker} {metric_name}: {value:.4f}")

CKA Evaluation: 100%|██████████| 115/115 [00:10<00:00, 11.35it/s]


  Similarity Stats:
    Average: 0.899477
    Highest: 0.993516
    Lowest:  0.724207
Merge Layer: 9


Computing importance (heads + FFN): 100%|██████████| 115/115 [00:13<00:00,  8.48it/s]


10 6
2246 1526
  Metrics:
      accuracy: 0.8775
    ★ f1: 0.9144


CKA Evaluation: 100%|██████████| 115/115 [00:09<00:00, 12.55it/s]


...

10 10
1998 2474
  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000


CKA Evaluation: 100%|██████████| 115/115 [00:05<00:00, 22.81it/s]


  Similarity Stats:
    Average: 0.855801
    Highest: 0.980212
    Lowest:  0.741874
Merge Layer: 5


Computing importance (heads + FFN): 100%|██████████| 115/115 [00:09<00:00, 11.88it/s]


12 16
2858 3014
  Metrics:
      accuracy: 0.3162
    ★ f1: 0.0000
